# 4. Pore analysis with Zeo++ (Windows edition)

Language: **English** | [中文](../cn/04_zeopp_pore_analysis.ipynb)

This native-Windows edition locates the TAPB–TFB CIF generated by the Windows edition of Notebook 2, runs cofkit's Zeo++ analysis wrapper from a regular Python cell, and prints human-readable pore information without requiring WSL. The equivalent `analyze_zeopp_pore_properties` Python workflow is included as commented code.

## Tutorial series

1. [Environment setup](01_environment_setup.ipynb)
2. [First COF build](02_first_cof_build.ipynb)
3. [Topology, connectivity, and linkage examples](03_topologies_connectivities_and_linkages.ipynb)
4. **Pore analysis with Zeo++** (this notebook)

Prerequisites: run the Windows edition of [Notebook 2](02_first_cof_build.ipynb) successfully and have a Windows-runnable Zeo++ 0.3 `network.exe` or `network` executable. The `cofkit` Python package itself works without Zeo++; only this analysis requires the external binary.

## Locate Notebook 2's CIF

The build pipeline groups CIFs by coarse validation class, so the exact subdirectory can vary. This search deliberately looks through all classes and fails with a useful message if Notebook 2 has not produced a CIF.

In [ ]:
from pathlib import Path

matches = sorted(
    Path("out/tutorial_02_first_build/cifs").rglob("tapb__tfb__hcb.cif")
)
if not matches:
    raise FileNotFoundError(
        "No TAPB-TFB CIF was found. Run Windows Notebook 2 before continuing."
    )

cif_path = matches[0].resolve()
print(f"Using CIF: {cif_path}\n")
print("\n".join(cif_path.read_text(encoding="utf-8").splitlines()[:18]))

In [ ]:
# Python equivalent (commented out): locate Notebook 2's CIF.
# from pathlib import Path
# matches = sorted(Path("out/tutorial_02_first_build/cifs").rglob("tapb__tfb__hcb.cif"))
# if not matches:
#     raise FileNotFoundError("Run Notebook 2 before continuing.")
# cif_path = matches[0]
# print(cif_path)

## Paste the Zeo++ path

In the next code cell, replace only `PASTE THE FULL PATH TO network.exe HERE` with the full path supplied by your instructor. Keep the surrounding quotation marks, then run the cell. For example, the edited line may look like `zeopp_path_text = r"C:\course-tools\zeo++-0.3\network.exe"`. The leading `r` keeps Windows backslashes literal.

The cell checks the file and saves the setting in the repository's local `.env` file, so you only need to paste it once. It preserves any other `.env` entries. Upstream Zeo++ 0.3 documents Windows compilation through Cygwin and GNU Make, but this notebook intentionally leaves installation to the instructor in restricted Windows environments.

In [ ]:
from pathlib import Path
import re

# Student: paste the full path inside this raw string.
zeopp_path_text = r"PASTE THE FULL PATH TO network.exe HERE"

placeholder = "PASTE THE FULL PATH TO network.exe HERE"
if zeopp_path_text == placeholder:
    raise ValueError(
        "Paste the full path to network.exe into zeopp_path_text, then rerun this cell."
    )

zeopp_path = Path(zeopp_path_text).expanduser()
if not zeopp_path.is_file():
    raise FileNotFoundError(f"Zeo++ executable does not exist: {zeopp_path}")
zeopp_path = zeopp_path.resolve()

repo_root = Path.cwd().resolve()
dotenv_path = repo_root / ".env"
existing_lines = (
    dotenv_path.read_text(encoding="utf-8-sig").splitlines()
    if dotenv_path.is_file()
    else []
)
kept_lines = [
    line
    for line in existing_lines
    if not re.match(r"\s*(?:export\s+)?COFKIT_ZEOPP_PATH\s*=", line)
]
kept_lines.append(f'COFKIT_ZEOPP_PATH="{zeopp_path}"')
dotenv_path.write_text("\n".join(kept_lines) + "\n", encoding="utf-8")
print(f"Zeo++ path saved for this repository: {zeopp_path}")

In [ ]:
# Zeo++ is an external C++ program. The runnable cell above records its path using
# Python's standard library; cofkit's direct Python API begins at the analysis step below.

## Run point-probe and finite-probe analysis

The baseline reports pore-limiting sphere sizes plus point-probe surface area and volume. The repeated `--probe-radius` values add accessibility-aware scans at illustrative radii of 1.20 Å and 1.86 Å. Adjust these radii to match the adsorbate model used in your own study.

The command intentionally omits `--json`: the terminal output is the concise human-readable report. A complete machine-readable `zeopp_report.json` is still written to disk.

In [ ]:
import os
from pathlib import Path
import shutil
import subprocess

repo_root = Path.cwd().resolve()
matches = sorted(
    Path("out/tutorial_02_first_build/cifs").rglob("tapb__tfb__hcb.cif")
)
if not matches:
    raise FileNotFoundError("Run Windows Notebook 2 before continuing.")
cif_path = matches[0].resolve()

dotenv_path = repo_root / ".env"
if not dotenv_path.is_file():
    raise FileNotFoundError("Run the Paste the Zeo++ path cell first.")
zeopp_entry = next(
    (
        line
        for line in dotenv_path.read_text(encoding="utf-8-sig").splitlines()
        if line.strip().startswith("COFKIT_ZEOPP_PATH=")
    ),
    None,
)
if zeopp_entry is None:
    raise RuntimeError("Run the Paste the Zeo++ path cell first.")
zeopp_path = Path(
    zeopp_entry.split("=", 1)[1].strip().strip("\"'")
).expanduser()
if not zeopp_path.is_file():
    raise FileNotFoundError(f"Zeo++ executable does not exist: {zeopp_path}")

conda = os.environ.get("CONDA_EXE") or shutil.which("conda")
if not conda:
    raise RuntimeError("Conda was not found. Run Windows Notebook 1 first.")
command = [
    conda, "run", "-n", "cofkit", "cofkit",
    "analyze", "zeopp", str(cif_path),
    "--zeopp-path", str(zeopp_path.resolve()),
    "--output-dir", "out/tutorial_04_zeopp",
    "--probe-radius", "1.20",
    "--probe-radius", "1.86",
    "--surface-samples-per-atom", "250",
    "--volume-samples-total", "5000",
]
subprocess.run(command, cwd=repo_root, check=True)

In [ ]:
# Python API equivalent (commented out), including a human-readable report.
# from pathlib import Path
# from cofkit import analyze_zeopp_pore_properties
#
# cif_path = next(Path("out/tutorial_02_first_build/cifs").rglob("tapb__tfb__hcb.cif"))
# zeopp_entry = next(
#     line for line in Path(".env").read_text(encoding="utf-8").splitlines()
#     if line.strip().startswith("COFKIT_ZEOPP_PATH=")
# )
# zeopp_binary = zeopp_entry.split("=", 1)[1].strip().strip("\"'")
# result = analyze_zeopp_pore_properties(
#     cif_path,
#     output_dir="out/tutorial_04_zeopp_python",
#     probe_radii=(1.20, 1.86),
#     surface_samples_per_atom=250,
#     volume_samples_total=5000,
#     zeopp_path=zeopp_binary,
# )
# basic = result.baseline.basic_pore_properties
# channels = result.baseline.point_probe_channels
# surface = result.baseline.point_probe_surface_area
# volume = result.baseline.point_probe_volume
# print(f"Input CIF: {result.input_cif}")
# print(f"Largest included sphere: {basic.largest_included_sphere:.3f} Å")
# print(f"Largest free sphere: {basic.largest_free_sphere:.3f} Å")
# print("Largest included sphere along a free path: "
#       f"{basic.largest_included_sphere_along_free_path:.3f} Å")
# print(f"Point-probe channels / pockets: {channels.n_channels} / {channels.n_pockets}")
# print(f"Point-probe accessible surface area: {surface.accessible_surface_area_a2:.3f} Å²")
# print(f"Point-probe accessible volume: {volume.accessible_volume_a3:.3f} Å³")
# for scan in result.probe_scans:
#     print(f"Probe radius {scan.settings.probe_radius:.2f} Å: {scan.status}")
#     if scan.surface_area is not None and scan.volume is not None:
#         print(f"  accessible surface area = {scan.surface_area.accessible_surface_area_a2:.3f} Å²")
#         print(f"  accessible volume fraction = {scan.volume.accessible_volume_fraction:.5f}")
#     if scan.accessibility is not None:
#         print(f"  accessible Voronoi-node fraction = {scan.accessibility.accessible_fraction}")
# print(f"Full report: {result.report_path}")

## How to read the main values

| Report field | Practical meaning |
|---|---|
| Largest included sphere | Diameter of the largest sphere that fits somewhere in the pore network |
| Largest free sphere | Diameter of the largest sphere that can traverse the periodic pore network |
| Largest included sphere along free path | Largest cavity encountered along a traversable path |
| Accessible surface area | Surface reachable by the selected probe under Zeo++'s geometric model |
| Accessible volume / fraction | Probe-accessible pore volume, either absolute or relative to the unit cell |
| Channels and pockets | Connected transport pathways versus isolated accessible regions |

All sphere sizes and probe settings in the report are in ångström-based units. Surface-area and volume estimates use Monte Carlo sampling, so increase the sample counts for production convergence studies.

## Inspect the retained report and raw outputs

cofkit preserves Zeo++ input/output artifacts and logs so the parsed report remains auditable.

In [ ]:
import json
from pathlib import Path

output_dir = Path("out/tutorial_04_zeopp")
report_path = output_dir / "zeopp_report.json"
if not report_path.is_file():
    raise FileNotFoundError("Run the Zeo++ analysis cell before inspecting its output.")

print("--- generated Zeo++ files ---")
print(*(path.resolve() for path in sorted(output_dir.rglob("*")) if path.is_file()), sep="\n")
print("\n--- zeopp_report.json ---")
report = json.loads(report_path.read_text(encoding="utf-8"))
print(json.dumps(report, indent=2))

In [ ]:
# Python equivalent (commented out): read the persisted report.
# import json
# from pathlib import Path
# report_path = Path("out/tutorial_04_zeopp/zeopp_report.json")
# report = json.loads(report_path.read_text())
# print(report["baseline"]["basic_pore_properties"])
# for scan in report["probe_scans"]:
#     print(scan["settings"]["probe_radius"], scan["status"])

## Before using the numbers

Pore descriptors are only as meaningful as the input geometry and atomic radii model. Treat this generated CIF as a tutorial input; for research calculations, optimize and validate the framework first, document the Zeo++ version and radii definitions, and check sampling convergence. A failed finite-probe scan is recorded in the JSON report without discarding successful baseline results.